# Lab 2: Encapsulation and Class Interaction
## IST1012 — Cybersecurity Fundamentals
**Difficulty: Medium** | **Estimated Time: 60 minutes**

In this lab you will deepen your OOP skills by learning about
**encapsulation** (protecting internal data), **private attributes**,
**getter/setter patterns**, and making **multiple objects interact**.
All examples are cybersecurity-themed.

## Learning Objectives
1. Use naming conventions to indicate private attributes
2. Implement getter and setter methods to control access
3. Validate data inside setter methods
4. Design classes that interact with each other
5. Understand why encapsulation matters in security software

---
## Part 1: Warm-Up (5-10 min)



**Q1.** Run the code below and observe what happens when we
modify an attribute directly. Is this safe?

In [1]:
class CryptoWallet:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

w = CryptoWallet("Alice", 100)
w.balance = -9999
print("Balance: " + str(w.balance))

Balance: -9999


---
## Part 2: Core Concepts (10-15 min)

### Private Attributes (Convention)

In Python, prefixing an attribute with an underscore (`_`) signals
that it should be treated as **internal / private**. A double
underscore (`__`) triggers name-mangling, making access harder.

This is a **convention**, not strict enforcement — but good
cybersecurity code respects it.

In [ ]:
class Credential:
    def __init__(self, username, password):
        self.username = username
        self._password = password   # convention: private

    def get_password_hint(self):
        return self._password[0] + "****"

cred = Credential("admin", "hunter2")
print(cred.get_password_hint())

# You CAN still access it, but you SHOULD NOT:
# print(cred._password)  # bad practice

### Getter and Setter Methods

Instead of allowing direct access, we provide methods that
**validate** before changing data.

In [ ]:
class SecureWallet:
    def __init__(self, owner, balance):
        self._owner = owner
        self._balance = balance

    def get_balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            print("Error: deposit must be positive")
            return
        self._balance = self._balance + amount

    def withdraw(self, amount):
        if amount <= 0:
            print("Error: withdrawal must be positive")
            return
        if amount > self._balance:
            print("Error: insufficient funds")
            return
        self._balance = self._balance - amount

wallet = SecureWallet("Bob", 500)
wallet.deposit(200)
print("Balance: " + str(wallet.get_balance()))
wallet.withdraw(1000)  # should show error

### Objects Working Together

Classes become powerful when objects interact.
For example, an `AuthSystem` might manage multiple `User` objects.

In [3]:
class User:
    def __init__(self, username, password):
        self.username = username
        self._password = password

    def verify(self, attempt):
        return attempt == self._password

class AuthSystem:
    def __init__(self):
        self._users = []

    def register(self, username, password):
        user = User(username, password)
        self._users.append(user)
        print("Registered: " + username)

    def login(self, username, password):
        for user in self._users:
            if user.username == username:
                if user.verify(password):
                    print("Login successful: " + username)
                    return True
                else:
                    print("Wrong password for: " + username)
                    return False
        print("User not found: " + username)
        return False

auth = AuthSystem()
auth.register("alice", "p@ssw0rd")
auth.register("bob", "12345")
auth.login("alice", "p@ssw0rd")
auth.login("alice", "wrong")
auth.login("eve", "hacker")

Registered: alice
Registered: bob
Login successful: alice
Wrong password for: alice
User not found: eve


False

---
## Part 3: Guided Exercises (25-30 min)

### Exercise 1 — Encapsulated UserProfile

Create a class `UserProfile` with:
- Private attributes `_username`, `_email`, and `_failed_logins` (starts at 0)
- A getter `get_username()` that returns the username
- A getter `get_email()` that returns the email
- A method `set_email(new_email)` that only updates the email
  if the new email contains an `@` symbol. Otherwise print an error.
- A method `record_failed_login()` that increments `_failed_logins` by 1.
- A method `is_locked()` that returns `True` if `_failed_logins >= 3`.

Test by creating a profile, setting a valid and invalid email,
recording several failed logins, and checking the lock status.

In [9]:
class UserProfile:
    def __init__(self, username, email):
        self._username = username
        self._email = email
        self._failed_logins = 0

    def get_username(self):
        return self._username

    def get_email(self):
        return self._email

    def set_email(self, new_email):
        if '@' in new_email:
            self._email = new_email
        else: print("Error!!! invalid email")

    def record_failed_login(self):
        self._failed_logins += 1

    def is_locked(self):
      if self._failed_logins >= 3:
        return True
      else:
        return False

In [16]:

profile = UserProfile("bersu","busu.com")

profile.set_email("new_email")
profile.record_failed_login()
profile.is_locked()

Error!!! invalid email


False

### Test Cases for Exercise 1: UserProfile

In [12]:
# Test Case 1: Basic Profile Creation and Getters
print("--- Test Case 1: Basic Profile Creation and Getters ---")
user_profile = UserProfile("john_doe", "john.doe@example.com")
print(f"Username: {user_profile.get_username()}")
print(f"Email: {user_profile.get_email()}")
print(f"Initial locked status: {user_profile.is_locked()}")
print(f"Initial failed logins: {user_profile._failed_logins}")

# Test Case 2: Set Valid Email
print("\n--- Test Case 2: Set Valid Email ---")
user_profile.set_email("john.doe.new@example.com")
print(f"New Email (valid): {user_profile.get_email()}")

# Test Case 3: Set Invalid Email
print("\n--- Test Case 3: Set Invalid Email ---")
user_profile.set_email("invalid-email.com") # Should print an error
print(f"Email after invalid attempt: {user_profile.get_email()}")

# Test Case 4: Record Failed Logins and Check Lock Status
print("\n--- Test Case 4: Record Failed Logins and Check Lock Status ---")
print(f"Failed logins before recording: {user_profile._failed_logins}")
print(f"Locked status before recording: {user_profile.is_locked()}")

user_profile.record_failed_login() # 1st failed login
print(f"After 1 failed login: Failed logins = {user_profile._failed_logins}, Locked = {user_profile.is_locked()}")

user_profile.record_failed_login() # 2nd failed login
print(f"After 2 failed logins: Failed logins = {user_profile._failed_logins}, Locked = {user_profile.is_locked()}")

user_profile.record_failed_login() # 3rd failed login (should lock)
print(f"After 3 failed logins: Failed logins = {user_profile._failed_logins}, Locked = {user_profile.is_locked()}")

user_profile.record_failed_login() # 4th failed login (still locked)
print(f"After 4 failed logins: Failed logins = {user_profile._failed_logins}, Locked = {user_profile.is_locked()}")


--- Test Case 1: Basic Profile Creation and Getters ---
Username: john_doe
Email: john.doe@example.com
Initial failed logins: 0

--- Test Case 2: Set Valid Email ---
New Email (valid): john.doe.new@example.com

--- Test Case 3: Set Invalid Email ---
Error!!! invalid email
Email after invalid attempt: john.doe.new@example.com

--- Test Case 4: Record Failed Logins and Check Lock Status ---
Failed logins before recording: 0
Locked status before recording: False
After 1 failed login: Failed logins = 1, Locked = False
After 2 failed logins: Failed logins = 2, Locked = False
After 3 failed logins: Failed logins = 3, Locked = True
After 4 failed logins: Failed logins = 4, Locked = True


### Exercise 2 — Token Bucket

In networking, a **token bucket** controls the rate of actions.
Create a class `TokenBucket` with:
- `__init__(self, capacity)` — sets `_tokens` to `capacity` and
  stores `_capacity`.
- `use_token()` — if tokens > 0, decrease by 1 and return `True`.
  Otherwise return `False`.
- `refill()` — reset `_tokens` to `_capacity`.
- `get_tokens()` — return current token count.

Simulate a scenario: create a bucket with capacity 3, use tokens
until empty, try one more (should fail), then refill.

In [21]:
class TokenBucket:
  def __init__(self,capacity):
    self._tokens = capacity
    self._capacity = capacity

    def use_token(self):
      if self.tokens > 0:
        self._tokens -= 1
        return True
      else:
        return False

    def refill(self):
      self._tokens = self._capacity   #capacity e esitliyor (reset)

    def get_tokens(self):
      return self._tokens


In [26]:
bucket = TokenBucket(3)

print("Tokens:"+ str(bucket.get_tokens()))
print(bucket.use_token())
print("Tokens:"+ str(bucket.get_tokens()))
bucket.refill()
print("Tokens:"+ str(bucket.get_tokens()))

AttributeError: 'TokenBucket' object has no attribute 'get_tokens'

### Exercise 3 — Firewall Rule + Firewall

Create **two classes** that work together:

**Class `FirewallRule`:**
- Attributes: `source_ip`, `dest_port`, `action` ("allow" or "deny")
- Method `matches(ip, port)` — returns `True` if both `ip` and `port` match.

**Class `Firewall`:**
- Stores a list of `FirewallRule` objects.
- Method `add_rule(rule)` — appends a rule.
- Method `check_packet(ip, port)` — loops through rules and returns
  the `action` of the first matching rule. If no rule matches, return "deny"
  (default deny policy).

Test with at least 3 rules and several packet checks.

In [27]:
class FirewallRule:
  def __init__(self,source_ip,dest_port,action):
    self.source_ip = source_ip
    self.dest_port = dest_port
    self.action = action

  def matches(self,ip,port):
    return self.source_ip == ip and self.dest_port == port

class Firewall:
  def __init__(self):
    self.rules = []

  def add_rule(self,rule):
    self.rules.append(rule)

  def check_packet(self,ip,port):
    for rule in self.rules:
      if rule.matches(ip, port):
        return rule.action
    return "deny"



---
## Part 4: Challenge Section (10 min)

### Challenge — Rate-Limited Login System

Combine concepts from this lab to build a `SecureAuthSystem`:
- Maintains a dictionary of registered users (username -> password).
- Maintains a dictionary of failed attempt counts per username.
- Method `register(username, password)` — adds a user.
- Method `login(username, password)`:
  - If the user does not exist, print an error.
  - If the user has 5+ failed attempts, print "Account locked" and
    refuse the login.
  - If the password is wrong, increment the fail counter and print a
    warning showing how many attempts remain.
  - If the password is correct, reset the fail counter and print success.

Test thoroughly: register users, trigger lockouts, successful logins.

In [28]:
class SecureAuthSystem:
    def __init__(self):
        self._users = {}
        self._failed_attempts = {}

    def register(self, username, password):
        self._users[username] = password
        self._failed_attempts[username] = 0
        print("Registered: " + username)

    def login(self, username, password):
        if username not in self._users:
            print("User not found: " + username)
            return  False

        if self._failed_attempts[username] >= 5:
            print("Account locked")
            return False

        if self._users[username] != password:
            self._failed_attempts[username] += 1
            print("Wrong password for: " + username)
            print(f"{5 - self._failed_attempts[username]} attempts remaining.")
            return False

        self._failed_attempts[username] = 0
        print("Login successful: " + username)
        return True

In [29]:
auth = SecureAuthSystem()
auth.register("alice", "p@ssw0rd")
auth.login("alice", "p@ssw0rd")
auth.login("alice", "wrong")
auth.login("eve", "hacker")


Registered: alice
Login successful: alice
Wrong password for: alice
4 attempts remaining.
User not found: eve


False

---
## Submission
- Save and submit your completed notebook.
- Ensure all cells run without errors.
- Submit via the course portal before the deadline.

**Well done — encapsulation is the first line of defense in good code!**